# GPS OD-flow Experiments (v8)

**Refactored notebook** — all model code imported from `models/GPS/` module.

## Experiment plan: GPS+TransFlower / LGBM ablations

**Base models:**
- B2: GPS+TransFlower, Huber, raw (CPC 0.681)
- B7: GPS+TransFlower, CE, normalized (CPC 0.699)
- C1: MC GPS+TransFlower, Huber, raw (CPC 0.387)
- C2: MC GPS+TransFlower, CE, normalized (CPC 0.466)

**Tier 1 (must run):** B7+log, B7+zinb, C2+log, C2+combo1, C1+sz30/50, C2+sz30/50
**Tier 2 (after Tier 1):** B2+log+spe, B7+sz30+log, B7+sz50+log, C1+log+gnorm, C2+spe+zinb
**Tier 3 (exploratory):** C1+log+spe+gnorm, B7+sz30+spe

In [ ]:
import sys, os, gc, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Project root
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Imports from models/GPS module ──
from models.GPS.config import (
    TrainingConfig, device, MC_EPOCHS,
    SINGLE_CITY_ID, MULTI_CITY_IDS,
    DATA_PATH, RESULTS_DIR, METRICS_CSV, WEIGHTS_DIR,
    save_metrics_to_csv, save_model_weights, ensure_dirs,
)
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.main import train_single_city, train_multi_city
from models.GPS.metrics import evaluate_full_matrix, compute_metrics
from models.GPS.lgbm_pipeline import train_lgbm_from_model

ensure_dirs()
print(f"Device: {device}")
print(f"Data path: {DATA_PATH}")
print(f"Results dir: {RESULTS_DIR}")

## Single-city experiments

Load data for city `48201`, then run experiments with pe_type caching.

In [ ]:
# Single-city data with pe_type caching
_scd_cache = {}

def get_single_city_data(pe_type='rwpe'):
    "Load and cache single-city data by pe_type."
    if pe_type not in _scd_cache:
        print(f"  Building single-city data pe_type={pe_type}...")
        _scd_cache[pe_type] = prepare_single_city_data(pe_type=pe_type)
    return _scd_cache[pe_type]

single_city_data = get_single_city_data('rwpe')
cd = single_city_data
print(f"City {cd['city_id']}: N={cd['num_nodes']}")
print(f"  train={cd['train_mask'].sum():,}  val={cd['val_mask'].sum():,}  test={cd['test_mask'].sum():,}")

In [ ]:
single_city_results = {}

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 1 — must run (filling critical gaps)
# ══════════════════════════════════════════════════════════════════════════════
single_city_runs = [
    # 1.1 B7+log: Log-transform on CE baseline.
    #     Log is the best single modification on B2 (+0.041) and C1 (+0.038).
    #     Never tested on B7 (strongest single-city baseline). Expected CPC ~0.73-0.74.
    ('B7+log', 'B7+Log: GPS+TF+CE+Log (norm)',
     TrainingConfig(decoder_type='transflower', loss_type='ce',
                    prediction_mode='normalized', use_dest_sampling=False,
                    use_log_transform=True)),

    # 1.2 B7+zinb: ZINB loss on CE baseline.
    #     Tested on B2 (neutral for TF, +0.008 for LGBM = best LGBM result overall).
    #     Never tested on B7.
    ('B7+zinb', 'B7+ZINB: GPS+TF+ZINB (norm)',
     TrainingConfig(decoder_type='transflower', loss_type='zinb',
                    prediction_mode='normalized', use_dest_sampling=False,
                    include_zero_pairs=True, zero_pair_ratio=0.5)),
]

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 2 — combining best single modifications (uncomment after Tier 1)
# ══════════════════════════════════════════════════════════════════════════════
# single_city_runs += [
#     # 2.1 B2+log+spe: Combine top-1 (log: +0.041) and top-2 (SPE: +0.015) for B2.
#     #     If additive, expected CPC ~0.74.
#     ('B2+log+spe', 'B2+Log+SPE: GPS+TF+Huber+Log+SPE (raw)',
#      TrainingConfig(decoder_type='transflower', loss_type='huber',
#                     prediction_mode='raw', use_log_transform=True, pe_type='spe')),
#
#     # 2.2 B7+sz30+log / B7+sz50+log: Combine two best single-city modifications.
#     #     sz30/sz50 addresses data sparsity, log addresses target scale — orthogonal.
#     #     Run only after B7+log (1.1) confirms log works on B7.
#     ('B7+sz30+log', 'B7+sz30+Log: GPS+TF+CE+sz30+Log',
#      TrainingConfig(decoder_type='transflower', loss_type='ce',
#                     prediction_mode='normalized', use_dest_sampling=True,
#                     n_dest_sample=128, include_zero_pairs=True, zero_pair_ratio=0.3,
#                     use_log_transform=True)),
#     ('B7+sz50+log', 'B7+sz50+Log: GPS+TF+CE+sz50+Log',
#      TrainingConfig(decoder_type='transflower', loss_type='ce',
#                     prediction_mode='normalized', use_dest_sampling=True,
#                     n_dest_sample=128, include_zero_pairs=True, zero_pair_ratio=0.5,
#                     use_log_transform=True)),
# ]

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 3 — exploratory (uncomment after Tier 2)
# ══════════════════════════════════════════════════════════════════════════════
# single_city_runs += [
#     # 3.2 B7+sz30+spe: zeros-sampling + SPE. Orthogonal mechanisms.
#     ('B7+sz30+spe', 'B7+sz30+SPE: GPS+TF+CE+sz30+SPE',
#      TrainingConfig(decoder_type='transflower', loss_type='ce',
#                     prediction_mode='normalized', use_dest_sampling=True,
#                     n_dest_sample=128, include_zero_pairs=True, zero_pair_ratio=0.3,
#                     pe_type='spe')),
# ]

print(f"Single-city runs: {len(single_city_runs)}")
for rid, rname, rcfg in single_city_runs:
    cd = get_single_city_data(rcfg.pe_type)
    single_city_results[rid] = train_single_city(rid, rname, rcfg, city_data=cd)

## LGBM on GPS embeddings

Train LightGBM on embeddings from each successfully trained single-city model.

In [ ]:
lgbm_results = {}

# Use each successful single-city model as embedding donor
for donor_id, donor_result in single_city_results.items():
    if donor_result.get('status') != 'ok':
        continue
    if donor_result.get('model') is None:
        continue
    lgbm_id = f"L_{donor_id}"
    lgbm_results[lgbm_id] = train_lgbm_from_model(
        lgbm_id, single_city_data, donor_result['model'], donor_id
    )

print(f"LGBM models trained: {len(lgbm_results)}")

## Multi-city experiments (10 cities)

Area-level split: 8 train, 1 val, 1 test. Data cached by pe_type.

In [ ]:
# Multi-city data with pe_type caching
_mcd_cache = {}

def get_mc_data(pe_type='rwpe'):
    "Load and cache multi-city data by pe_type."
    if pe_type not in _mcd_cache:
        print(f"  Building MC data pe_type={pe_type}...")
        cdd, tr, vl, te = prepare_multi_city_data(pe_type=pe_type)
        _mcd_cache[pe_type] = (cdd, tr, vl, te)
    return _mcd_cache[pe_type]

cdd, train_ids, val_ids, test_ids = get_mc_data('rwpe')
print(f"Train: {train_ids}")
print(f"Val:   {val_ids}")
print(f"Test:  {test_ids}")
for cid in sorted(cdd.keys()):
    print(f"  {cid}: N={cdd[cid]['num_nodes']}")

In [ ]:
multi_city_results = {}

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 1 — must run (filling critical gaps)
# ══════════════════════════════════════════════════════════════════════════════
mc_runs = [
    # 1.3 C2+log: Log-transform on MC CE baseline.
    #     C2 is the strongest MC model (0.466). Log consistently helps (+0.038-0.041).
    ('C2+log', 'C2+Log: MC GPS+TF+CE+Log (norm)',
     TrainingConfig(decoder_type='transflower', loss_type='ce',
                    prediction_mode='normalized', use_dest_sampling=True,
                    include_zero_pairs=False, use_log_transform=True,
                    mc_epochs=MC_EPOCHS)),

    # 1.4 C2+combo1 (SPE+GraphNorm): Super-linear synergy on C1 (+0.071).
    #     Critical to check whether synergy reproduces on CE base.
    ('C2+combo1', 'C2+SPE+GN: MC GPS+TF+CE+SPE+GN',
     TrainingConfig(decoder_type='transflower', loss_type='ce',
                    prediction_mode='normalized', use_dest_sampling=True,
                    include_zero_pairs=False, pe_type='spe',
                    gps_norm_type='graph_norm', mc_epochs=MC_EPOCHS)),

    # 1.5 C1+sz30 and C1+sz50: zeros-sampling on MC Huber.
    #     Best modification for single-city B7 (+0.026-0.027). Never tested on MC.
    ('C1+sz30', 'C1+sz30: MC GPS+TF+Huber+sz30',
     TrainingConfig(decoder_type='transflower', loss_type='huber',
                    prediction_mode='raw', use_dest_sampling=True,
                    n_dest_sample=128, include_zero_pairs=True,
                    zero_pair_ratio=0.3, mc_epochs=MC_EPOCHS)),
    ('C1+sz50', 'C1+sz50: MC GPS+TF+Huber+sz50',
     TrainingConfig(decoder_type='transflower', loss_type='huber',
                    prediction_mode='raw', use_dest_sampling=True,
                    n_dest_sample=128, include_zero_pairs=True,
                    zero_pair_ratio=0.5, mc_epochs=MC_EPOCHS)),

    # 1.6 C2+sz30 and C2+sz50: zeros-sampling on MC CE.
    ('C2+sz30', 'C2+sz30: MC GPS+TF+CE+sz30',
     TrainingConfig(decoder_type='transflower', loss_type='ce',
                    prediction_mode='normalized', use_dest_sampling=True,
                    n_dest_sample=128, include_zero_pairs=True,
                    zero_pair_ratio=0.3, mc_epochs=MC_EPOCHS)),
    ('C2+sz50', 'C2+sz50: MC GPS+TF+CE+sz50',
     TrainingConfig(decoder_type='transflower', loss_type='ce',
                    prediction_mode='normalized', use_dest_sampling=True,
                    n_dest_sample=128, include_zero_pairs=True,
                    zero_pair_ratio=0.5, mc_epochs=MC_EPOCHS)),
]

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 2 — combining best single modifications (uncomment after Tier 1)
# ══════════════════════════════════════════════════════════════════════════════
# mc_runs += [
#     # 2.3 C1+log+gnorm: Top-1 (GraphNorm: +0.042) and top-2 (log: +0.038) on C1.
#     #     Combo1 (SPE+GN) showed super-linear synergy on C1, suggesting GN has
#     #     good combinatorial properties.
#     ('C1+log+gnorm', 'C1+Log+GN: MC GPS+TF+Huber+Log+GN',
#      TrainingConfig(decoder_type='transflower', loss_type='huber',
#                     prediction_mode='raw', use_log_transform=True,
#                     gps_norm_type='graph_norm', mc_epochs=MC_EPOCHS)),
#
#     # 2.4 C2+spe+zinb (Combo2): SPE+ZINB on MC CE.
#     #     Combo2 gave +0.022 on C1. Consider only if C2+combo1 (1.4) shows positive.
#     ('C2+spe+zinb', 'C2+SPE+ZINB: MC GPS+TF+ZINB+SPE',
#      TrainingConfig(decoder_type='transflower', loss_type='zinb',
#                     prediction_mode='normalized', use_dest_sampling=True,
#                     include_zero_pairs=True, zero_pair_ratio=0.5,
#                     pe_type='spe', mc_epochs=MC_EPOCHS)),
# ]

# ══════════════════════════════════════════════════════════════════════════════
#  Tier 3 — exploratory (uncomment after Tier 2)
# ══════════════════════════════════════════════════════════════════════════════
# mc_runs += [
#     # 3.1 C1+log+spe+gnorm (triple combination):
#     #     If C1+log+gnorm works, add SPE. SPE alone hurts C1, but GraphNorm
#     #     appears to "stabilize" it (combo1: +0.071). Log may further amplify.
#     ('C1+log+spe+gnorm', 'C1+Log+SPE+GN: MC GPS+TF+Huber+Log+SPE+GN',
#      TrainingConfig(decoder_type='transflower', loss_type='huber',
#                     prediction_mode='raw', use_log_transform=True,
#                     pe_type='spe', gps_norm_type='graph_norm',
#                     mc_epochs=MC_EPOCHS)),
#
#     # 3.3 Scale model for MC: If CPC gap > 0.15, try 2x width or depth.
#     # ('C_best_2x', 'C_best_2x: MC GPS+TF scaled', ...),
# ]

print(f"MC runs: {len(mc_runs)}")
for rid, rname, rcfg in mc_runs:
    mcd, tr, vl, _ = get_mc_data(rcfg.pe_type)
    multi_city_results[rid] = train_multi_city(
        rid, rname, rcfg, city_data_dict=mcd,
        train_city_ids=tr, val_city_ids=vl)

## Results

In [ ]:
def print_summary(rd, title):
    if not rd:
        print(f"  {title}: none")
        return
    print(f"\n{'='*130}\n  {title}\n{'='*130}")
    print(f"  {'ID':<18} {'Name':<50} {'Loss':<8} {'PE':<6} {'Norm':<12} "
          f"{'CPC_f':>7} {'CPC_nz':>7} {'CPC_t':>7} {'MAE':>8} {'Status':<12}")
    bc = max((r['metrics_nonzero']['CPC'] for r in rd.values()
              if r.get('metrics_nonzero', {}).get('CPC', 0) > 0), default=0)
    for k in sorted(rd.keys()):
        r = rd[k]
        mf = r.get('metrics_full', {})
        mnz = r.get('metrics_nonzero', {})
        mt = r.get('metrics_test_pairs', {})
        cfg = r.get('config', TrainingConfig())
        st = r.get('status', 'ok')
        m = ' <-' if abs(mnz.get('CPC', 0) - bc) < 1e-6 else ''
        print(f"  {k:<18} {r['name']:<50} {cfg.loss_type:<8} {cfg.pe_type:<6} "
              f"{cfg.gps_norm_type:<12} {mf.get('CPC',0):>7.4f} {mnz.get('CPC',0):>7.4f} "
              f"{mt.get('CPC',0):>7.4f} {mf.get('MAE',0):>8.4f} {st:<12}{m}")
    print(f"{'='*130}")

print_summary(single_city_results, f"Single-city | {SINGLE_CITY_ID}")
print_summary(lgbm_results, "LGBM (single-city donors)")
print_summary(multi_city_results, "Multi-city")

# CSV summary
if METRICS_CSV.exists():
    print(f"\n  All metrics saved to: {METRICS_CSV}")
    print(pd.read_csv(METRICS_CSV).to_string(index=False))

In [ ]:
def plot_histories(results, title):
    wh = {k: v for k, v in results.items()
           if 'history' in v and v.get('status') == 'ok'}
    if not wh:
        return
    n = len(wh)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    for ri, (rid, r) in enumerate(wh.items()):
        h = r['history']
        ep = range(1, len(h['train_loss']) + 1)
        axes[ri, 0].plot(ep, h['train_loss'])
        axes[ri, 0].set_title(f"{rid}: Train Loss")
        axes[ri, 0].grid(True, alpha=0.3)
        axes[ri, 1].plot(ep, h['val_loss'], color='orange')
        axes[ri, 1].set_title(f"{rid}: Val Loss")
        axes[ri, 1].grid(True, alpha=0.3)
        axes[ri, 2].plot(ep, h['val_cpc_full'], label='Full', color='green')
        axes[ri, 2].plot(ep, h['val_cpc_nz'], label='NZ', color='blue', ls='--')
        axes[ri, 2].set_title(f"{rid}: CPC")
        axes[ri, 2].set_ylim(0, 1)
        axes[ri, 2].legend()
        axes[ri, 2].grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_histories(single_city_results, "Single-city training")
plot_histories(multi_city_results, "Multi-city training")